# DS2002 · Walmart Warm-Up SKU Mess

**Lab — 2026-10-16 · Fall 2026**  

---

## Lab 08 — Walmart Warm-Up (SKU Mess)

This lab uses the **real midterm data** in `../06-midterm-walmart/data/`. It's a warm-up so you hit the ground running on the project. On Kaggle/Colab, upload the data folder or adjust the path.

*If the file isn't found, the cell falls back to a small inline sample so the notebook still runs.*

In [10]:
import pandas as pd

cat = pd.read_csv('/content/product_catalog.csv')

cat.head()

,sku,product_name,category,subcategory,unit_cost,supplier,package_size
0,PT-12,pop-tarts strawberry,Breakfast Foods,NaN,1.93,Johnson & Johnson,4-pack
1,SKU#459812,Strawberry Pop-Tart,Breakfast,Pop-Tarts,2.26,Roku Inc,4pk
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,NaN,2.11,Sara Lee,6 pack
3,PT_STRAWBERRY,pop-tarts strawberry,breakfast,Pop-Tarts,NaN,Campbell's,4 pack
4,WTR-24,BOTTLED WATER 24PK,Emergency Supplies,Bottled,2.64,Sara Lee,1 pack


### Q1 — How many rows look like Pop-Tart products?

Filter on `product_name` containing `pop`, case-insensitively. Print the count and the matching rows.

In [11]:
pop_rows = cat[
    cat['product_name'].str.contains('pop', case=False, na=False)
]

print("Number of Pop-Tart rows:", len(pop_rows))
print(pop_rows[['sku', 'product_name', 'category']])

Number of Pop-Tart rows: 4
             sku          product_name         category
0          PT-12  pop-tarts strawberry  Breakfast Foods
1     SKU#459812   Strawberry Pop-Tart        Breakfast
2  POPTART-STRAW  Pop-Tarts Strawberry        Breakfast
3  PT_STRAWBERRY  pop-tarts strawberry        breakfast


### Q2 — Which SKUs and names all mean *strawberry* Pop-Tart?

List them. Then note which Pop-Tart rows are **not** strawberry — those must stay separate, and saying so is part of the answer.

In [12]:
strawberry_pop = pop_rows[
    pop_rows['product_name'].str.contains('straw', case=False, na=False)
]

print("Strawberry Pop-Tart SKUs and names:")
print(strawberry_pop[['sku', 'product_name']])

other_pop = pop_rows[
    ~pop_rows['product_name'].str.contains('straw', case=False, na=False)
]

print("\nOther Pop-Tart products:")
print(other_pop[['sku', 'product_name']])

Strawberry Pop-Tart SKUs and names:
             sku          product_name
0          PT-12  pop-tarts strawberry
1     SKU#459812   Strawberry Pop-Tart
2  POPTART-STRAW  Pop-Tarts Strawberry
3  PT_STRAWBERRY  pop-tarts strawberry

Other Pop-Tart products:
Empty DataFrame
Columns: [sku, product_name]
Index: []


**Pop-Tart rows that are a different product:** None. All four Pop-Tart rows are strawberry Pop-Tarts, so they can be consolidated.

### Q3 — Build the mapping

A dict from each variant SKU to one canonical SKU. Then apply it in a way that leaves unmapped SKUs as themselves rather than turning them into `NaN` — that single mistake silently deletes most of the catalog.

In [13]:
sku_mapping = {
    'PT-12': 'PT-12',
    'SKU#459812': 'PT-12',
    'POPTART-STRAW': 'PT-12',
    'PT_STRAWBERRY': 'PT-12'
}

cat['canonical_sku'] = cat['sku'].map(sku_mapping).fillna(cat['sku'])

cat[['sku', 'product_name', 'canonical_sku']]

,sku,product_name,canonical_sku
0,PT-12,pop-tarts strawberry,PT-12
1,SKU#459812,Strawberry Pop-Tart,PT-12
2,POPTART-STRAW,Pop-Tarts Strawberry,PT-12
3,PT_STRAWBERRY,pop-tarts strawberry,PT-12
4,WTR-24,BOTTLED WATER 24PK,WTR-24
5,WATER24PK,Bottled Water 24-Pack,WATER24PK
6,WTR_24PK,Water - 24 Pack,WTR_24PK
7,BATT-100,BATTERIES AA 4PK,BATT-100
8,BAT-AA4,AA Batteries 4-Pack,BAT-AA4
9,BATT_AA_4PK,AA Batteries 4-Pack,BATT_AA_4PK


### Q4 — Normalize `category`

Strip and title-case, then show the distinct values before and after with counts. Note any pair that is still two labels for one thing — `Breakfast` and `Breakfast Foods` are not the same string but they are the same category.

In [14]:
print("Before normalization:")
print(cat['category'].value_counts(dropna=False))

cat['category_normalized'] = cat['category'].str.strip().str.title()

print("\nAfter normalization:")
print(cat['category_normalized'].value_counts(dropna=False))

Before normalization:
category
Emergency              5
Emergency Supplies     3
EMERGENCY              3
ENTERTAINMENT          3
GROCERY                3
Breakfast              2
entertainment          2
household              2
ELECTRONICS            2
health                 2
Baby                   2
Electronics            1
ALCOHOL                1
Alcoholic Beverages    1
breakfast              1
Breakfast Foods        1
Groceries              1
Grocery                1
electronics            1
snacks                 1
grocery                1
HOUSEHOLD              1
Snacks                 1
Household              1
baby                   1
Baby Products          1
emergency              1
Name: count, dtype: int64

After normalization:
category_normalized
Emergency              9
Entertainment          5
Grocery                5
Electronics            4
Household              4
Emergency Supplies     3
Breakfast              3
Baby                   3
Snacks                 2
H

### Q5 — Find the fragmentation you have not spotted yet

Hand-scanning a catalog does not scale. **TODO:** build a candidate list programmatically — normalize each product name (lowercase, strip punctuation, sort the words) and group by that, then show any group holding more than one SKU.

This is the workflow from Monday's lecture, and it is how you will actually approach the full catalog on the midterm.

In [15]:
import re

def normalize_name(name):
    words = re.sub(r'[^\w\s]', ' ', str(name).lower()).split()
    return ' '.join(sorted(words))

cat['normalized_name'] = cat['product_name'].apply(normalize_name)

candidates = (
    cat.groupby('normalized_name')
       .agg(
           sku_count=('sku', 'nunique'),
           skus=('sku', list),
           product_names=('product_name', list)
       )
)

candidates = candidates[candidates['sku_count'] > 1]

candidates

,sku_count,skus,product_names
normalized_name,,,
24 beer light pack,2,"[BEER-24, LTBEER24]","[Light Beer (24-pack), Light Beer (24-pack)]"
4 aa batteries pack,2,"[BAT-AA4, BATT_AA_4PK]","[AA Batteries 4-Pack, AA Batteries 4-Pack]"
aid first kit,2,"[FAID-01, FIRSTAID-KIT]","[First-Aid Kit, First Aid Kit]"
board game monopoly,3,"[BG-MONO, MONOPOLY-01, GAME-MONO]","[Monopoly Board Game, Monopoly Board Game, Boa..."
bread white,2,"[BREAD-WH, WHITEBREAD]","[WHITE BREAD, WHITE BREAD]"
butter peanut,2,"[PB-16, PNUTBTR-16]","[PEANUT BUTTER, Peanut Butter]"
candles emergency,2,"[CNDL-EM, CANDLE-EMRG]","[Candles (Emergency), Emergency Candles]"
canned soup,2,"[SOUP-CAN, CNSOUP-01]","[Soup - Canned, Canned Soup]"
cards playing,2,"[CARDS-01, PLAYCARD-STD]","[Playing Cards, PLAYING CARDS]"


### Q6 — Prove the mapping is safe

**TODO:** assert that applying your mapping produced no nulls in the canonical column and did not change the number of rows. Consolidation renames; it never adds or removes.

In [16]:
original_row_count = len(cat)

cat['canonical_sku'] = cat['sku'].map(sku_mapping).fillna(cat['sku'])

assert cat['canonical_sku'].notna().all()
assert len(cat) == original_row_count

print("Mapping is safe: no nulls and row count is unchanged.")

Mapping is safe: no nulls and row count is unchanged.


### Write-up

**a)** Name two product families beyond Pop-Tarts that appear fragmented, and how you would group them.

**b)** Where does consolidation stop being a data-cleaning task and become a business decision? Give one example from this catalog where you genuinely are not sure, and say who you would ask.

**a)** Two product families beyond Pop-Tarts that appear fragmented are Monopoly board games and first-aid kits. The Monopoly SKUs BG-MONO, MONOPOLY-01, and GAME-MONO all have the same product name, so I would group them under one canonical SKU. The first-aid kit SKUs FAID-01 and FIRSTAID-KIT also refer to the same product and should be grouped under one canonical SKU.

**b)** Consolidation will be taken into account in case there is insufficient information to ascertain that similar products are one and the same. This means that while “Alcohol” and “Alcoholic Beverages” might have been grouped under one category, consolidation will not be made without ascertaining the intent of the business in classifying the items. I would ask the product or catalog manager for clarification.